In [13]:
import os
import ast
import pandas as pd


def split_ndcg_summary_per_fold_wide(
    output_dir="hasil_eval_ndcg",
    k_folds=5,
):
    """
    Memisahkan hasil summary_ndcg.csv menjadi file CSV terpisah per fold,
    dengan format lebar (pivot): setiap nilai k menjadi kolom tersendiri
    (misal ndcg@10, ndcg@20, ...), berisi skor NDCG untuk kombinasi
    k_user & k_item pada fold tersebut.
    """
    # save_dir = os.path.join(output_dir, "ndcg_hybrid_kr")
    save_dir = os.path.join(output_dir, "ndcg_hybrid_EV")
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")

    if not os.path.exists(summary_path):
        print(f"File tidak ditemukan: {summary_path}")
        return

    per_fold_dir = os.path.join(save_dir, "per_fold_ndcg_wide")
    os.makedirs(per_fold_dir, exist_ok=True)

    print(f"Membaca ringkasan dari: {summary_path}")
    df_summary = pd.read_csv(summary_path)

    if "fold_scores" not in df_summary.columns:
        print("Kolom 'fold_scores' tidak ditemukan di summary_ndcg.csv")
        return

    def parse_fold_scores(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x if isinstance(x, list) else []

    df_summary["fold_scores"] = df_summary["fold_scores"].apply(parse_fold_scores)

    # Membangun baris long-format dulu: k_user, k_item, k, fold, ndcg
    rows = []
    for _, row in df_summary.iterrows():
        fold_scores = row["fold_scores"]
        for i, ndcg_value in enumerate(fold_scores):
            fold = i + 1
            if fold > k_folds:
                continue
            rows.append(
                {
                    "k_user": row["k_user"],
                    "k_item": row["k_item"],
                    "k": row["k"],
                    "fold": fold,
                    "ndcg": ndcg_value,
                }
            )

    df_long = pd.DataFrame(rows)

    if df_long.empty:
        print("Tidak ada data untuk diproses.")
        return

    # Untuk setiap fold, pivot: baris = (k_user, k_item), kolom = k, isi = ndcg
    for fold in range(1, k_folds + 1):
        df_fold = df_long[df_long["fold"] == fold]

        df_pivot = df_fold.pivot_table(
            index=["k_user", "k_item"],
            columns="k",
            values="ndcg",
        ).reset_index()

        # Rename kolom k menjadi format ndcg@k, misal ndcg@10, ndcg@20, dst
        df_pivot.columns = [
            f"ndcg@{col}" if isinstance(col, (int, float)) else col
            for col in df_pivot.columns
        ]

        fold_path = os.path.join(per_fold_dir, f"ndcg_fold_{fold}_wide.csv")
        df_pivot.to_csv(fold_path, index=False)
        print(f"  Hasil fold {fold} (format lebar) disimpan di: {fold_path}")

    print(f"Selesai. Semua hasil per fold (wide format) disimpan di folder: {per_fold_dir}")

In [14]:
def split_ndcg_summary_per_fold_wide_ub(
    output_dir="hasil_eval_ndcg_ML1M",
    k_folds=5,
):
    """
    Memisahkan hasil summary_ndcg.csv menjadi file CSV terpisah per fold,
    dengan format lebar (pivot): setiap nilai k menjadi kolom tersendiri
    (misal ndcg@10, ndcg@20, ...), berisi skor NDCG untuk kombinasi
    k_user & k_item pada fold tersebut.
    """
    # save_dir = os.path.join(output_dir, "ndcg_ub_kr")
    save_dir = os.path.join(output_dir, "ndcg_ub_ev")
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")

    if not os.path.exists(summary_path):
        print(f"File tidak ditemukan: {summary_path}")
        return

    per_fold_dir = os.path.join(save_dir, "per_fold_ndcg_wide")
    os.makedirs(per_fold_dir, exist_ok=True)

    print(f"Membaca ringkasan dari: {summary_path}")
    df_summary = pd.read_csv(summary_path)

    if "fold_scores" not in df_summary.columns:
        print("Kolom 'fold_scores' tidak ditemukan di summary_ndcg.csv")
        return

    def parse_fold_scores(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x if isinstance(x, list) else []

    df_summary["fold_scores"] = df_summary["fold_scores"].apply(parse_fold_scores)

    # Membangun baris long-format dulu: k_user, k_item, k, fold, ndcg
    rows = []
    for _, row in df_summary.iterrows():
        fold_scores = row["fold_scores"]
        for i, ndcg_value in enumerate(fold_scores):
            fold = i + 1
            if fold > k_folds:
                continue
            rows.append(
                {
                    "k_item": row["k_item"],
                    "k": row["k"],
                    "fold": fold,
                    "ndcg": ndcg_value,
                }
            )

    df_long = pd.DataFrame(rows)

    if df_long.empty:
        print("Tidak ada data untuk diproses.")
        return

    # Untuk setiap fold, pivot: baris = (k_user, k_item), kolom = k, isi = ndcg
    for fold in range(1, k_folds + 1):
        df_fold = df_long[df_long["fold"] == fold]

        df_pivot = df_fold.pivot_table(
            index=["k_item"],
            columns="k",
            values="ndcg",
        ).reset_index()

        # Rename kolom k menjadi format ndcg@k, misal ndcg@10, ndcg@20, dst
        df_pivot.columns = [
            f"ndcg@{col}" if isinstance(col, (int, float)) else col
            for col in df_pivot.columns
        ]

        fold_path = os.path.join(per_fold_dir, f"ndcg_fold_{fold}_wide.csv")
        df_pivot.to_csv(fold_path, index=False)
        print(f"  Hasil fold {fold} (format lebar) disimpan di: {fold_path}")

    print(f"Selesai. Semua hasil per fold (wide format) disimpan di folder: {per_fold_dir}")

In [15]:
def split_ndcg_summary_per_fold_wide_ib(
    output_dir="hasil_eval_ndcg",
    k_folds=5,
):
    """
    Memisahkan hasil summary_ndcg.csv menjadi file CSV terpisah per fold,
    dengan format lebar (pivot): setiap nilai k menjadi kolom tersendiri
    (misal ndcg@10, ndcg@20, ...), berisi skor NDCG untuk kombinasi
    k_user & k_item pada fold tersebut.
    """
    # save_dir = os.path.join(output_dir, "ndcg_ib_kr")
    save_dir = os.path.join(output_dir, "ndcg_ib_ev")
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")

    if not os.path.exists(summary_path):
        print(f"File tidak ditemukan: {summary_path}")
        return

    per_fold_dir = os.path.join(save_dir, "per_fold_ndcg_wide")
    os.makedirs(per_fold_dir, exist_ok=True)

    print(f"Membaca ringkasan dari: {summary_path}")
    df_summary = pd.read_csv(summary_path)

    if "fold_scores" not in df_summary.columns:
        print("Kolom 'fold_scores' tidak ditemukan di summary_ndcg.csv")
        return

    def parse_fold_scores(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x if isinstance(x, list) else []

    df_summary["fold_scores"] = df_summary["fold_scores"].apply(parse_fold_scores)

    # Membangun baris long-format dulu: k_user, k_item, k, fold, ndcg
    rows = []
    for _, row in df_summary.iterrows():
        fold_scores = row["fold_scores"]
        for i, ndcg_value in enumerate(fold_scores):
            fold = i + 1
            if fold > k_folds:
                continue
            rows.append(
                {
                    "k_user": row["k_user"],
                    "k": row["k"],
                    "fold": fold,
                    "ndcg": ndcg_value,
                }
            )

    df_long = pd.DataFrame(rows)

    if df_long.empty:
        print("Tidak ada data untuk diproses.")
        return

    # Untuk setiap fold, pivot: baris = (k_user, k_item), kolom = k, isi = ndcg
    for fold in range(1, k_folds + 1):
        df_fold = df_long[df_long["fold"] == fold]

        df_pivot = df_fold.pivot_table(
            index=["k_user",],
            columns="k",
            values="ndcg",
        ).reset_index()

        # Rename kolom k menjadi format ndcg@k, misal ndcg@10, ndcg@20, dst
        df_pivot.columns = [
            f"ndcg@{col}" if isinstance(col, (int, float)) else col
            for col in df_pivot.columns
        ]

        fold_path = os.path.join(per_fold_dir, f"ndcg_fold_{fold}_wide.csv")
        df_pivot.to_csv(fold_path, index=False)
        print(f"  Hasil fold {fold} (format lebar) disimpan di: {fold_path}")

    print(f"Selesai. Semua hasil per fold (wide format) disimpan di folder: {per_fold_dir}")

In [16]:
split_ndcg_summary_per_fold_wide(output_dir="hasil_eval_ndcg", k_folds=5)

Membaca ringkasan dari: hasil_eval_ndcg\ndcg_hybrid_EV\summary_ndcg.csv
  Hasil fold 1 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide\ndcg_fold_1_wide.csv
  Hasil fold 2 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide\ndcg_fold_2_wide.csv
  Hasil fold 3 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide\ndcg_fold_3_wide.csv
  Hasil fold 4 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide\ndcg_fold_4_wide.csv
  Hasil fold 5 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide\ndcg_fold_5_wide.csv
Selesai. Semua hasil per fold (wide format) disimpan di folder: hasil_eval_ndcg\ndcg_hybrid_EV\per_fold_ndcg_wide


In [17]:
split_ndcg_summary_per_fold_wide_ub(output_dir="hasil_eval_ndcg", k_folds=5)

Membaca ringkasan dari: hasil_eval_ndcg\ndcg_ub_ev\summary_ndcg.csv
  Hasil fold 1 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide\ndcg_fold_1_wide.csv
  Hasil fold 2 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide\ndcg_fold_2_wide.csv
  Hasil fold 3 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide\ndcg_fold_3_wide.csv
  Hasil fold 4 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide\ndcg_fold_4_wide.csv
  Hasil fold 5 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide\ndcg_fold_5_wide.csv
Selesai. Semua hasil per fold (wide format) disimpan di folder: hasil_eval_ndcg\ndcg_ub_ev\per_fold_ndcg_wide


In [18]:
split_ndcg_summary_per_fold_wide_ib(output_dir="hasil_eval_ndcg", k_folds=5)

Membaca ringkasan dari: hasil_eval_ndcg\ndcg_ib_ev\summary_ndcg.csv
  Hasil fold 1 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide\ndcg_fold_1_wide.csv
  Hasil fold 2 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide\ndcg_fold_2_wide.csv
  Hasil fold 3 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide\ndcg_fold_3_wide.csv
  Hasil fold 4 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide\ndcg_fold_4_wide.csv
  Hasil fold 5 (format lebar) disimpan di: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide\ndcg_fold_5_wide.csv
Selesai. Semua hasil per fold (wide format) disimpan di folder: hasil_eval_ndcg\ndcg_ib_ev\per_fold_ndcg_wide
